# LrcSSM -- Liquid-Resistance Liquid-Capacitance State-Space Model

Farsang, Hasani, Rus, Grosu, *Parallelization of Non-linear State-Space Models: Scaling Up Liquid-Resistance Liquid-Capacitance Networks for Efficient Sequence Modeling*, NeurIPS 2025 ([arXiv:2505.21717](https://arxiv.org/abs/2505.21717)) -- the newest model in this repo.

A non-linear, input-dependent recurrence whose Jacobian is diagonal by construction, so the whole sequence can be solved with a parallel scan instead of a sequential loop. See `model.py` (`LrcSSMLayer`, both `parallel=True/False` paths) and `../../papers/README.md`.

This notebook (1) checks the sequential and parallel-scan paths agree, (2) times both, and (3) trains on ETTh1 forecasting.

In [ ]:
import sys, time
sys.path.insert(0, '../..')
sys.path.insert(0, '.')

import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from liquid_playground.data import load_ett
from liquid_playground.device import resolve_device
from liquid_playground.utils.seed import set_seed
from model import LrcSSMModel

set_seed(0)
device = resolve_device('auto')  # or 'cpu' / 'cuda' / 'mps'
print('device:', device)

In [ ]:
seq_len, pred_len = 96, 24
train_x, train_y, test_x, test_y = load_ett(seq_len=seq_len, pred_len=pred_len)
n_channels = train_x.shape[-1]
train_x, test_x = train_x.to(device), test_x.to(device)
train_y_flat = train_y.reshape(train_y.shape[0], -1).to(device)
test_y_flat = test_y.reshape(test_y.shape[0], -1).to(device)
print(train_x.shape, train_y_flat.shape)

In [ ]:
model = LrcSSMModel(input_size=n_channels, state_size=32, output_size=pred_len * n_channels).to(device)

with torch.no_grad():
    sample = train_x[:8]
    seq_out = model.layers[0](sample, parallel=False)
    par_out = model.layers[0](sample, parallel=True)
    print('sequential vs. parallel-scan max abs diff:', (seq_out - par_out).abs().max().item())

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

batch_size = 64
n_train = train_x.shape[0]
history = {'train_mse': [], 'test_mse': []}
epochs = 15
t0 = time.time()
for epoch in range(epochs):
    model.train()
    perm = torch.randperm(n_train, device=device)
    total = 0.0
    for i in range(0, n_train, batch_size):
        idx = perm[i:i + batch_size]
        opt.zero_grad()
        pred = model(train_x[idx], parallel=True)
        loss = loss_fn(pred, train_y_flat[idx])
        loss.backward()
        opt.step()
        total += loss.item() * len(idx)
    model.eval()
    with torch.no_grad():
        test_mse = loss_fn(model(test_x, parallel=True), test_y_flat).item()
    history['train_mse'].append(total / n_train)
    history['test_mse'].append(test_mse)
print(f'total train time: {time.time() - t0:.1f}s, final test MSE: {history["test_mse"][-1]:.4f}')

In [ ]:
plt.plot(history['train_mse'], label='train')
plt.plot(history['test_mse'], label='test')
plt.title('LrcSSM MSE'); plt.xlabel('epoch'); plt.legend()
plt.show()